## Scenario 2: A cross-functional team with one data scientist working on an ML model


MLflow setup:
- tracking server: yes, local server
- backend store: sqlite database
- artifacts store: local filesystem

The experiments can be explored locally by accessing the local tracking server.

To run this example you need to launch the mlflow server locally by running the following command in your terminal:

`mlflow server --backend-store-uri sqlite:///backend.db`

In [2]:
import mlflow


mlflow.set_tracking_uri("http://127.0.0.1:5000")

In [3]:
print(f"tracking URI: '{mlflow.get_tracking_uri()}'")

tracking URI: 'http://127.0.0.1:5000'


In [4]:
mlflow.search_experiments()

[<Experiment: artifact_location='mlflow-artifacts:/0', creation_time=1792949075323, effective_trace_archival_retention=None, experiment_id='0', last_update_time=1792949075323, lifecycle_stage='active', name='Default', tags={}, trace_location=None, workspace='default'>]

In [5]:
from sklearn.linear_model import LogisticRegression
from sklearn.datasets import load_iris
from sklearn.metrics import accuracy_score

mlflow.set_experiment("my-experiment-1")

with mlflow.start_run():

    X, y = load_iris(return_X_y=True)

    params = {"C": 0.1, "random_state": 42}
    mlflow.log_params(params)

    lr = LogisticRegression(**params).fit(X, y)
    y_pred = lr.predict(X)
    mlflow.log_metric("accuracy", accuracy_score(y, y_pred))

    mlflow.sklearn.log_model(lr, artifact_path="models")
    print(f"default artifacts URI: '{mlflow.get_artifact_uri()}'")

2026/10/25 18:28:08 INFO mlflow.tracking.fluent: Experiment with name 'my-experiment-1' does not exist. Creating a new experiment.
2026/10/25 18:28:08 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/10/25 18:28:14 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


default artifacts URI: 'mlflow-artifacts:/1/fae707e50ffc48428c381e54e974d41b/artifacts'
🏃 View run thundering-elk-889 at: http://127.0.0.1:5000/#/experiments/1/runs/fae707e50ffc48428c381e54e974d41b
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


In [6]:
mlflow.search_experiments()

[<Experiment: artifact_location='mlflow-artifacts:/1', creation_time=1792949288111, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1792949288111, lifecycle_stage='active', name='my-experiment-1', tags={}, trace_location=None, workspace='default'>,
 <Experiment: artifact_location='mlflow-artifacts:/0', creation_time=1792949075323, effective_trace_archival_retention=None, experiment_id='0', last_update_time=1792949075323, lifecycle_stage='active', name='Default', tags={}, trace_location=None, workspace='default'>]

### Interacting with the model registry

In [7]:
from mlflow.tracking import MlflowClient


client = MlflowClient("http://127.0.0.1:5000")

In [8]:
client.search_registered_models()

[]

In [9]:
run_id = client.search_runs(experiment_ids='1')[0].info.run_id
mlflow.register_model(
    model_uri=f"runs:/{run_id}/models",
    name='iris-classifier'
)

Successfully registered model 'iris-classifier'.
2026/10/25 18:30:11 WARNING mlflow.tracking._model_registry.fluent: Run with id fae707e50ffc48428c381e54e974d41b has no artifacts at artifact path 'models', registering model based on models:/m-59e7ca1a736147948d4e3d8d89629925 instead
2026/10/25 18:30:12 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: iris-classifier, version 1
Created version '1' of model 'iris-classifier'.


<ModelVersion: aliases=[], creation_timestamp=1792949411975, current_stage='None', deployment_job_state=None, description='', last_updated_timestamp=1792949411975, metrics=None, model_id=None, name='iris-classifier', params=None, run_id='fae707e50ffc48428c381e54e974d41b', run_link='', source='models:/m-59e7ca1a736147948d4e3d8d89629925', status='READY', status_message=None, tags={}, user_id='', version='1', workspace='default'>

In [10]:
client.search_registered_models()

[<RegisteredModel: aliases={}, creation_timestamp=1792949411545, deployment_job_id='', deployment_job_state='DEPLOYMENT_JOB_CONNECTION_STATE_UNSPECIFIED', description='', last_updated_timestamp=1792949411975, latest_versions=[<ModelVersion: aliases=[], creation_timestamp=1792949411975, current_stage='None', deployment_job_state=None, description='', last_updated_timestamp=1792949411975, metrics=None, model_id=None, name='iris-classifier', params=None, run_id='fae707e50ffc48428c381e54e974d41b', run_link='', source='models:/m-59e7ca1a736147948d4e3d8d89629925', status='READY', status_message=None, tags={}, user_id='', version='1', workspace='default'>], name='iris-classifier', tags={}, workspace='default'>]